### Fórmulas de Optimización de ALBERT vs. BERT
#### 1. Factorización de la Matriz de Incrustación (Factorized Embedding Parametrization)
En BERT tradicional, el tamaño del vocabulario $V$ está drectamente acoplado al tamaño de la capa oculta $H$, lo que genera una complejidad de parámetros de:
$$P_{\text{BERT}} = V \times H$$

ALBERT rompe esta dependencia dividiendo la matriz en dos pasos de menor dimensión utilizando un espacio intermedio $E$ (donde $E \ll H$, típicamente $E=128$ y $H=768$).
$$P_{\text{ALBERT}} = (V \times E) + (E \times H)$$

...

#### 2. Compartición de Parámetros entre Capas (Cross-Layer Parameter Sharing)
$$h_l = \text{LayerNorm}(h_{l-1} + \text{SubLayer}(h_{l-1}; \Theta)) $$






In [ ]:
# 3. Código en Python
# Está optimizado para procesar textos en lote (batches),
# enviar los cálculos a la GPU si está disponible, calcular las probabilidades exactas
# y extraer los resultados en un formato estructurado.

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd

# 1. Configuración de hardware (GPU acelerada si está disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilizando dispositivo: {device}")

# 2. Inicializar ALBERT (Base v2) y asignarle la salida para clasificar categorías
model_name = "albert/albert-base-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Supongamos 4 tipos de flujos de reclamos en la aseguradora
categorias_seguros = ["Auto/Vía Pública", "Hogar/Estructura", "Gastos Médicos", "Fraude/Sospechoso"]
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(categorias_seguros))
model.to(device)

# 3. Datos de entrada (Reclamaciones reales de clientes)
textos_siniestros = [
    "Hubo una inundación en mi cocina debido a la rotura de una tubería del fregadero.",
    "Un auto me chocó por detrás en el semáfoo de la avenida principal y se dio a la fuga",
    "El asegurado presenta facturas médicas duplicadas de clínicas distintas en las misma fecha."
]

# 4. Tokenización en lote (Fórmula de procesamiento)
inputs = tokenizer(
    textos_siniestros,
    padding=True, # Rellena los textos cortos para que midan lo mismo
    truncation=True, # Corta los textos que pasen del límite
    max_length=128, # Tamaño óptimo de secuencia
    return_tensors="pt" # Devuelve tensores de PyTorch
).to(device)

# 5. Inferencia (Pso por el Encoder de ALBERT)
model.eval()
with torch.no_grad():
    outputs = model(**inputs)

# 6. Post-procesamiento matemático (Fórmula Softmax aplicada a la dimensión final)
logits = outputs.logits
probabilidades = torch.nn.functional.softmax(logits, dim=-1)

# 7. Formatear y presentar los resultados en un DataFrame limpio
resultados = []
for i, texto in enumerate(textos_siniestros):
    probs_lista = probabilidades[i].pu().tolist()
    idx_max = torch.argmax(probabilidades[i]).item()

    fila = {
        "Texto del Cliente": texto[:60] + "...",
        "Categoría Predicaha": categorias_seguros[idx_max],
        "Confianza": f"{probs_lista[idx_max] * 100:.2f}%"
    }
    resultados.append(fila)

df_analisis = pd.DataFrame(resultados)
print("\n", df_analisis.to_string(index=False))
